In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

super_ai_engineer_season_6_ocr_2569_round2_path = kagglehub.competition_download('super-ai-engineer-season-6-ocr-2569-round2')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569-round2/final_data/submission_template_v4.csv
/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569-round2/final_data/images/constituency_22_3_page2.png
/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569-round2/final_data/images/constituency_31_9.png
/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569-round2/final_data/images/party_list_13_1_page3.png
/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569-round2/final_data/images/party_list_16_3_page3.png
/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569-round2/final_data/images/party_list_10_33_page3.png
/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569-round2/final_data/images/constituency_10_28.png
/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569-round2/final_data/images/party_list_31_5_page4.png
/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569-round2/final_data/images/party_list

In [ ]:
!pip install pillow
!pip install opencv-python
!pip install pytesseract
!pip install -q -U google-generativeai
!pip install thefuzz
!apt-get update
!apt-get install -y tesseract-ocr tesseract-ocr-tha
!pip install pytesseract
!apt-get install tesseract-ocr-tha
!pip install pythainlp
!pip install thefuzz


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease   
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,618 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,173 kB]
Fetched 6,175 kB in 3s (2,105 kB/s)                          
Reading package lists... Done
W: Skipping acquire of config

In [ ]:
import os
import cv2
import numpy as np
import pytesseract
import re
import pandas as pd
from tqdm.notebook import tqdm # สำหรับ Progress Bar ใน Kaggle

from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt

import shutil # สำหรับ Copy ไฟล์


In [ ]:
from pythainlp.tokenize import word_tokenize
from thefuzz import process

กรองรุปภาพ ใช้รูปแค่ที่มี ตาราง

In [ ]:

import shutil # สำหรับ Copy ไฟล์

# 1. ตั้งค่าพาร์ท
input_path = '/kaggle/input'
output_dir = 'cleaned_data'

# สร้าง Folder ใหม่ถ้ายังไม่มี
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"สร้างโฟลเดอร์ '{output_dir}' เรียบร้อยแล้ว")

# 2. เริ่มการสแกนหาไฟล์
for dirname, _, filenames in os.walk(input_path):
    for filename in sorted(filenames):
        # กรองเฉพาะไฟล์รูปภาพ
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            full_path = os.path.join(dirname, filename)

            # อ่านภาพเป็น Grayscale
            img = cv2.imread(full_path, cv2.IMREAD_GRAYSCALE)
            if img is None: continue

            # --- ขั้นตอนการเช็คว่ามีตารางหรือไม่ ---
            # ทำ Threshold แบบ Invert
            thresh = cv2.threshold(img, 200, 255, cv2.THRESH_BINARY_INV)[1]

            # หาเส้นแนวนอน
            horiz_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
            horiz_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horiz_kernel, iterations=2)

            # หาเส้นแนวตั้ง
            vert_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
            vert_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vert_kernel, iterations=2)

            # หาจุดตัดของเส้น (Intersection) เพื่อยืนยันว่าเป็นตาราง
            # ถ้ามีจุดที่เส้นแนวตั้งและแนวนอนทับกันเยอะพอ แสดงว่าเป็นตาราง
            intersections = cv2.bitwise_and(horiz_lines, vert_lines)

            # นับจำนวนจุดตัด (ถ้ามีจุดตัด > 10 จุด ให้ถือว่าเป็นรูปที่มีตาราง)
            num_intersections = np.sum(intersections > 0)

            if num_intersections > 10:
                # 3. คัดแยก: Copy ไฟล์ต้นฉบับไปที่โฟลเดอร์เป้าหมาย
                target_path = os.path.join(output_dir, filename)
                shutil.copy(full_path, target_path)
                print(f"[คัดแยกสำเร็จ] พบตารางใน: {filename} (จุดตัด: {num_intersections})")
            else:
                # ไม่ใช่ตาราง หรือตารางไม่ชัดเจน
                pass

print(f"\nเสร็จสิ้น! คุณสามารถเข้าไปเช็ครูปที่มีตารางได้ในโฟลเดอร์: {output_dir}")

สร้างโฟลเดอร์ 'cleaned_data' เรียบร้อยแล้ว
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_1.png (จุดตัด: 31)
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_10.png (จุดตัด: 34)
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_10_page2.png (จุดตัด: 1103)
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_11.png (จุดตัด: 222)
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_11_page2.png (จุดตัด: 1775)
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_12.png (จุดตัด: 32)
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_12_page2.png (จุดตัด: 1204)
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_12_page3.png (จุดตัด: 28)
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_13.png (จุดตัด: 35)
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_13_page2.png (จุดตัด: 1074)
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_14.png (จุดตัด: 31)
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_14_page2.png (จุดตัด: 1311)
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_14_page3.png (จุดตัด: 94)
[คัดแยกสำเร็จ] พบตารางใน: constituency_10_16.png (จุดตัด: 44)
[คัดแยกสำเร็จ] พบตารางใน: constituenc

In [ ]:
source_dir = '/kaggle/working/cleaned_data'
final_results = []


files_to_process = [f for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
for filename in tqdm(sorted(files_to_process), desc="กำลังอ่านคะแนน"):
    print(os.path.join(source_dir, filename))

กำลังอ่านคะแนน:   0%|          | 0/825 [00:00<?, ?it/s]

/kaggle/working/cleaned_data/constituency_10_1.png
/kaggle/working/cleaned_data/constituency_10_10.png
/kaggle/working/cleaned_data/constituency_10_10_page2.png
/kaggle/working/cleaned_data/constituency_10_11.png
/kaggle/working/cleaned_data/constituency_10_11_page2.png
/kaggle/working/cleaned_data/constituency_10_12.png
/kaggle/working/cleaned_data/constituency_10_12_page2.png
/kaggle/working/cleaned_data/constituency_10_12_page3.png
/kaggle/working/cleaned_data/constituency_10_13.png
/kaggle/working/cleaned_data/constituency_10_13_page2.png
/kaggle/working/cleaned_data/constituency_10_14.png
/kaggle/working/cleaned_data/constituency_10_14_page2.png
/kaggle/working/cleaned_data/constituency_10_14_page3.png
/kaggle/working/cleaned_data/constituency_10_16.png
/kaggle/working/cleaned_data/constituency_10_16_page2.png
/kaggle/working/cleaned_data/constituency_10_17.png
/kaggle/working/cleaned_data/constituency_10_17_page2.png
/kaggle/working/cleaned_data/constituency_10_18.png
/kaggle/wor

In [ ]:
# for dirname, _, filenames in os.walk('/kaggle/working/cleaned_data'):
#     for filename in sorted(filenames):
#         if filename == "constituency_10_10_page2.png":
#             full_path = os.path.join(dirname, filename)


## แปลงรูปภาพเป็นข้อมูล
source_dir = '/kaggle/working/cleaned_data'
final_results = []


files_to_process = [f for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]



# 3. เริ่มประมวลผลพร้อม Progress Bar
files_to_process = sorted(files_to_process, key=lambda x: (
    re.sub(r'_page\d+', '', x),
    int(re.search(r'_page(\d+)', x).group(1)) if re.search(r'_page(\d+)', x) else 0
))

for filename in tqdm(files_to_process, desc="กำลังอ่านคะแนน"):
    full_path = os.path.join(source_dir, filename)

    base_name = os.path.splitext(filename)[0]
    # ตัดคำว่า _page... ออก (ถ้ามี) เช่น constituency_10_1_page2 -> constituency_10_1
    clean_name = re.sub(r'_page\d+', '', base_name)

    img_gray = cv2.imread(full_path, cv2.IMREAD_GRAYSCALE)
    img_color = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2BGR)

    # 1. สร้างตารางขาวดำ (Inverted)
    thresh = cv2.threshold(img_gray, 200, 255, cv2.THRESH_BINARY_INV)[1]

    # 2. ตรวจจับโครงสร้างตาราง (เส้นตั้ง-นอน)
    horiz_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    detect_horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horiz_kernel, iterations=2)

    vert_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
    detect_vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vert_kernel, iterations=2)

    table_structure = cv2.addWeighted(detect_horizontal, 0.5, detect_vertical, 0.5, 0.0)
    table_structure = cv2.threshold(table_structure, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]

    # 3. หา Contours ทั้งหมดภายในตาราง
    contours, _ = cv2.findContours(table_structure, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    # 4. คัดกรองเฉพาะกล่องที่มีขนาดเหมาะสมและเก็บพิกัด
    cells = []
    from collections import defaultdict
    file_counters = defaultdict(int)
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w > 50 and h > 20:  # กรอง noise เล็กๆ ออก
            cells.append((x, y, w, h))

    # 5. จัดกลุ่ม Cells ให้เป็นแถว (Rows)
    # เรียงจากบนลงล่างตามค่า y
    cells = sorted(cells, key=lambda b: b[1])
    rows = []
    if cells:
        current_row = [cells[0]]
        for i in range(1, len(cells)):
            # ถ้าค่า y ต่างกันไม่เกิน 15px ให้ถือว่าแถวเดียวกัน
            if abs(cells[i][1] - current_row[-1][1]) <= 15:
                current_row.append(cells[i])
            else:
                # เรียงจากซ้ายไปขวาในแถวนั้นๆ
                rows.append(sorted(current_row, key=lambda b: b[0]))
                current_row = [cells[i]]
        rows.append(sorted(current_row, key=lambda b: b[0]))

    # 6. เลือกเฉพาะแถวข้อมูล (ข้ามแถวสุดท้าย)
    data_rows = rows[1:-1]

    # # 7. วนลูปอ่านเฉพาะ ของแต่ละแถวที่เลือก


    print(f"--- เริ่มดึงข้อมูลเฉพาะข้อความในวงเล็บ ---")

    for idx, row in enumerate(data_rows):
        for cell in row:   # วนทุกช่อง
            x, y, w, h = cell

            # debug
            cv2.rectangle(img_color, (x, y), (x + w, y + h), (0, 0, 255), 2)

            # crop + preprocess
            cell_img = img_gray[y:y+h, x:x+w]
            cell_img = cv2.resize(cell_img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
            _, cell_thresh = cv2.threshold(cell_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

            # OCR
            raw_text = pytesseract.image_to_string(
                cell_thresh, lang='tha', config='--psm 6'
            ).strip()

            if re.search(r"รวม\s*คะแนน", raw_text):
                break

            # ดึงเฉพาะ "ข้อความในวงเล็บ"
            match = re.search(r"\(\s*([ก-์\s]+)", raw_text)
            if match:
                        thai_word = match.group(1).strip()
                        # print(thai_word)
                        score_int = thai_word_to_int(thai_word)

                        file_counters[clean_name] += 1
                        record = f"{clean_name}_{file_counters[clean_name]} : {score_int}"
                        print(record)
                        final_results.append(record)

                        break



    # โชว์ภาพที่มาร์คตำแหน่งที่อ่านไว้
    # cv2_imshow(img_color)

กำลังอ่านคะแนน:   0%|          | 0/825 [00:00<?, ?it/s]

--- เริ่มดึงข้อมูลเฉพาะข้อความในวงเล็บ ---
--- เริ่มดึงข้อมูลเฉพาะข้อความในวงเล็บ ---
--- เริ่มดึงข้อมูลเฉพาะข้อความในวงเล็บ ---
constituency_10_10_1 : 11800
constituency_10_10_2 : 19017
constituency_10_10_3 : 9110
constituency_10_10_4 : 7915
constituency_10_10_6 : 6372
constituency_10_10_7 : 2012
constituency_10_10_8 : 1137
constituency_10_10_9 : 636
constituency_10_10_10 : 583
constituency_10_10_12 : 503
constituency_10_10_13 : 195
constituency_10_10_14 : 161
constituency_10_10_15 : 282
constituency_10_10_16 : 203
constituency_10_10_17 : 168
--- เริ่มดึงข้อมูลเฉพาะข้อความในวงเล็บ ---
constituency_10_11_1 : 0
--- เริ่มดึงข้อมูลเฉพาะข้อความในวงเล็บ ---
constituency_10_11_1 : 24850
constituency_10_11_2 : 20118
constituency_10_11_3 : 1810
constituency_10_11_4 : 1100
constituency_10_11_5 : 0
constituency_10_11_6 : 110
constituency_10_11_8 : 186
constituency_10_11_9 : 120
constituency_10_11_10 : 504
constituency_10_11_11 : 400
constituency_10_11_12 : 310
constituency_10_11_13 : 12
constitu

In [ ]:
## เอาไว้ทดสอบไฟล์เดียว
for dirname, _, filenames in os.walk('/kaggle/working/cleaned_data'):
    for filename in sorted(filenames):
        if filename == "party_list_33.png":
            full_path = os.path.join(dirname, filename)

            base_name = os.path.splitext(filename)[0]
            # ตัดคำว่า _page... ออก (ถ้ามี) เช่น constituency_10_1_page2 -> constituency_10_1
            clean_name = re.sub(r'_page\d+', '', base_name)

            img_gray = cv2.imread(full_path, cv2.IMREAD_GRAYSCALE)
            img_color = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2BGR)

            # 1. สร้างตารางขาวดำ (Inverted)
            thresh = cv2.threshold(img_gray, 200, 255, cv2.THRESH_BINARY_INV)[1]

            # 2. ตรวจจับโครงสร้างตาราง (เส้นตั้ง-นอน)
            horiz_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
            detect_horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horiz_kernel, iterations=2)

            vert_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
            detect_vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vert_kernel, iterations=2)

            table_structure = cv2.addWeighted(detect_horizontal, 0.5, detect_vertical, 0.5, 0.0)
            table_structure = cv2.threshold(table_structure, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]

            # 3. หา Contours ทั้งหมดภายในตาราง
            contours, _ = cv2.findContours(table_structure, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

            # 4. คัดกรองเฉพาะกล่องที่มีขนาดเหมาะสมและเก็บพิกัด
            cells = []
            for cnt in contours:
                x, y, w, h = cv2.boundingRect(cnt)
                if w > 50 and h > 20:  # กรอง noise เล็กๆ ออก
                    cells.append((x, y, w, h))

            # 5. จัดกลุ่ม Cells ให้เป็นแถว (Rows)
            # เรียงจากบนลงล่างตามค่า y
            cells = sorted(cells, key=lambda b: b[1])
            rows = []
            if cells:
                current_row = [cells[0]]
                for i in range(1, len(cells)):
                    # ถ้าค่า y ต่างกันไม่เกิน 15px ให้ถือว่าแถวเดียวกัน
                    if abs(cells[i][1] - current_row[-1][1]) <= 15:
                        current_row.append(cells[i])
                    else:
                        # เรียงจากซ้ายไปขวาในแถวนั้นๆ
                        rows.append(sorted(current_row, key=lambda b: b[0]))
                        current_row = [cells[i]]
                rows.append(sorted(current_row, key=lambda b: b[0]))

            # 6. เลือกเฉพาะแถวข้อมูล (ข้ามแถวสุดท้าย)
            data_rows = rows[1:]

            # 7. วนลูปอ่าน แต่ละแถวที่เลือก

            print(f"--- เริ่มดึงข้อมูลเฉพาะข้อความในวงเล็บ ---")

            for idx, row in enumerate(data_rows):
                for cell in row:   # วนทุกช่อง
                    x, y, w, h = cell

                    # debug
                    cv2.rectangle(img_color, (x, y), (x + w, y + h), (0, 0, 255), 2)

                    # crop + preprocess
                    cell_img = img_gray[y:y+h, x:x+w]
                    cell_img = cv2.resize(cell_img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
                    _, cell_thresh = cv2.threshold(cell_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

                    # OCR
                    raw_text = pytesseract.image_to_string(
                        cell_thresh, lang='tha', config='--psm 6'
                    ).strip()

                    if re.search(r"รวม\s*คะแนน", raw_text):
                        break

                    # ดึงเฉพาะ "ข้อความในวงเล็บ"
                    match = re.search(r"\(\s*([ก-์\s]+)", raw_text)
                    # print(raw_text)

                    if match:
                        thai_word = match.group(1).strip()
                        # print(thai_word)
                        score_int = thai_word_to_int(thai_word)

                        record = f"{clean_name}_{idx+1} : {score_int}"
                        print(record)
                        final_results.append(record)

                        break



            # โชว์ภาพที่มาร์คตำแหน่งที่อ่านไว้
            cv2_imshow(img_color)

In [ ]:
# แปลเป็นตัวเลข
def thai_word_to_int(text):
    if not text:
        return 0

    thai_nums = {
        'ศูนย์':0, 'หนึ่ง':1, 'เอ็ด':1, 'สอง':2, 'ยี่':2,
        'สาม':3, 'สี่':4, 'ห้า':5, 'หก':6, 'เจ็ด':7, 'แปด':8, 'เก้า':9
    }

    units = [
        ('ล้าน', 1000000),
        ('แสน', 100000),
        ('หมื่น', 10000),
        ('พัน', 1000),
        ('ร้อย', 100),
        ('สิบ', 10)
    ]

    text = text.replace("หมืน", "หมื่น")
    text = re.sub(r'[^\u0E00-\u0E7F]', '', text)

    total = 0

    for unit_name, unit_val in units:
        if unit_name in text:
            parts = text.split(unit_name, 1)
            prefix = parts[0]

            digit = 1  # default เช่น "สิบ"
            for word, val in thai_nums.items():
                if prefix.endswith(word):
                    digit = val
                    break

            total += digit * unit_val
            text = parts[1]

    # ตัวสุดท้าย
    for word, val in thai_nums.items():
        if text.endswith(word):
            total += val
            break

    return total

In [ ]:
# เช็คค่า
final_results

['constituency_10_10_1 : 11800',
 'constituency_10_10_2 : 19017',
 'constituency_10_10_3 : 9110',
 'constituency_10_10_4 : 7915',
 'constituency_10_10_6 : 6372',
 'constituency_10_10_7 : 2012',
 'constituency_10_10_8 : 1137',
 'constituency_10_10_9 : 636',
 'constituency_10_10_10 : 583',
 'constituency_10_10_12 : 503',
 'constituency_10_10_13 : 195',
 'constituency_10_10_14 : 161',
 'constituency_10_10_15 : 282',
 'constituency_10_10_16 : 203',
 'constituency_10_10_17 : 168',
 'constituency_10_11_1 : 0',
 'constituency_10_11_1 : 24850',
 'constituency_10_11_2 : 20118',
 'constituency_10_11_3 : 1810',
 'constituency_10_11_4 : 0',
 'constituency_10_11_5 : 0',
 'constituency_10_11_6 : 110',
 'constituency_10_11_8 : 186',
 'constituency_10_11_9 : 120',
 'constituency_10_11_10 : 504',
 'constituency_10_11_11 : 400',
 'constituency_10_11_12 : 310',
 'constituency_10_11_13 : 12',
 'constituency_10_11_14 : 6',
 'constituency_10_11_16 : 0',
 'constituency_10_11_17 : 100',
 'constituency_10_11_1

In [ ]:
# 1. แปลง List ของ String ให้เป็น List ของ Dictionary เพื่อแยกคอลัมน์
formatted_data = []
for item in final_results:
    # แยก ID (ชื่อไฟล์_ลำดับ) ออกจาก Score ด้วย " : "
    if " : " in item:
        parts = item.split(" : ")
        formatted_data.append({
            "Id": parts[0].strip(),
            "Votes": parts[1].strip()
        })

# 2. สร้าง DataFrame (ตาราง)
df = pd.DataFrame(formatted_data)

# 3. บันทึกเป็นไฟล์ CSV
# encoding='utf-8-sig'
output_file = 'sample_submission_V3.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"--- Export สำเร็จ! ---")
print(f"บันทึกไฟล์ไปที่: {os.path.abspath(output_file)}")

# โชว์ตัวอย่างตาราง 5 แถวแรก
df.head()

--- Export สำเร็จ! ---
บันทึกไฟล์ไปที่: /kaggle/working/sample_submission_V3.csv


,Id,Votes
0,constituency_10_10_1,11800
1,constituency_10_10_2,19017
2,constituency_10_10_3,9110
3,constituency_10_10_4,7915
4,constituency_10_10_6,6372
